ML_sweep/
├── trials/
│   ├── trial_000/
│   │   ├── M_001_L_013/
│   │   ├── M_004_L_013/
│   │   └── ...
│   ├── trial_001/
│   └── ...
├── ML_sweep_trial_metrics.csv
├── ML_sweep_metrics_mean_std.csv
└── aggregate_functions/

In [1]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "src").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not locate project root containing src/")


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

SRC = PROJECT_ROOT / "src"
ARTIFACTS = PROJECT_ROOT / "artifacts"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CWD:", Path.cwd())
print("ARTIFACTS:", ARTIFACTS)

PROJECT_ROOT: .
CWD: .
ARTIFACTS: ./artifacts


In [2]:
# --------------------------------------------------
# Repeated-learning configuration
# --------------------------------------------------
N_TRIALS = 10
BASE_SEED = 909
TRIAL_SEED_STRIDE = 100_000_000

# False allows interrupted sweeps to resume from saved cases.
# Use True only when intentionally regenerating every dataset and fit.
OVERWRITE = True

print("N_TRIALS:", N_TRIALS)
print("BASE_SEED:", BASE_SEED)
print("TRIAL_SEED_STRIDE:", TRIAL_SEED_STRIDE)
print("OVERWRITE:", OVERWRITE)

N_TRIALS: 10
BASE_SEED: 909
TRIAL_SEED_STRIDE: 100000000
OVERWRITE: True


# Levy

In [3]:
SYSTEM_NAME = "levi"
PROFILE_NAME = "levi_velocity_grid"
INTERACTION_TYPE = "alignment"
SYSTEM_ORDER = 2
FIT_END_TIME = 0.75
PREDICTION_END_TIME = 2.25
TRAIN_BEGIN_TIME = 0.25

M_VALUES = (1, 4, 16, 64, 256)
L_VALUES = (51, 101, 201, 401, 801)
# M_VALUES = (20,)
# L_VALUES = (51,)

In [4]:
from ips_learning.analysis.ml_sweep_runner import MLSweepConfig, MLSweepRunner

In [5]:
config_6C = MLSweepConfig(
    system_name=SYSTEM_NAME,
    profile_name=PROFILE_NAME,
    interaction_type=INTERACTION_TYPE,
    system_order=SYSTEM_ORDER,
    artifact_root=ARTIFACTS,

    n_trials=N_TRIALS,
    base_seed=BASE_SEED,
    trial_seed_stride=TRIAL_SEED_STRIDE,

    N=10,
    D=2,
    t0=TRAIN_BEGIN_TIME,

    M_values=M_VALUES,
    L_values=L_VALUES,

    fit_horizon_end=FIT_END_TIME,
    prediction_horizon_end=PREDICTION_END_TIME,

    radial_degree=1,
    radial_n_basis=8,
    environment_degree=1,
    environment_n_basis_per_dim=10,

    solver_method="lstsq",
    ridge_lambda=1e-8,
    svd_tol=1e-12,

    kernel_grid_size=500,
    environment_grid_size=120,

    overwrite=OVERWRITE,
)

In [6]:
# print("System:", config_6C.system_name)
# print("Number of trials:", config_6C.n_trials)
# print("Base seed:", config_6C.base_seed)
# print("Trial seed stride:", config_6C.trial_seed_stride)
# print("Overwrite:", config_6C.overwrite)
# print(
#     "Expected trial rows:",
#     config_6C.n_trials
#     * len(config_6C.M_values)
#     * len(config_6C.L_values),
# )

In [7]:
# --------------------------------------------------
# Run repeated-trial M/L sweep: Levy
# --------------------------------------------------
levi_runner = MLSweepRunner(config_6C)

# One row per (trial_index, M, L).
levi_trial_df = levi_runner.run()

levi_eval_dir = (
    Path(config_6C.artifact_root)
    / config_6C.system_name
    / "evaluations"
    / "ML_sweep"
)

levi_summary_path = (
    levi_eval_dir
    / "ML_sweep_metrics_mean_std.csv"
)

# One row per (M, L), containing mean/std across trials.
levi_summary_df = pd.read_csv(levi_summary_path)

print("Trial-level rows:", len(levi_trial_df))
print("Summary rows:", len(levi_summary_df))

display(levi_trial_df.head())
display(levi_summary_df.head())

Starting trial_000: 1/10
Trial base seed: 909
[1/250] trial_000, M=1, L=51
Generating/loading trial-specific dataset: trial_000, M=1, L=51
Running trial-specific case: trial_000, M=1, L=51
fit_end_index:51
T_fit:   0.75
t:   (51,)
X:   (1, 51, 10, 2)
V: (1, 51, 10, 2)
LHS: (1, 51, 10, 2)
U0:  (1, 40)
interaction type: alignment
condition number: 1.2499950784124628e+16
rank: 208
basis localization: {'system_name': 'levi', 'system_order': 2, 'interaction_type': 'alignment', 'basis_stage': '2.2', 'environment_variable': 'velocity', 'r_min': 0.0, 'r_max': 2.151561528652411, 'env_min': [-1.0, -1.0], 'env_max': [1.0, 1.0], 'radial_family': 'bspline', 'environment_family': 'bspline', 'radial_degree': 1, 'environment_degree': 1, 'n_radial_basis': 8, 'n_energy_basis': 8, 'n_alignment_basis': 8, 'energy_basis_source': 'radial_basis', 'alignment_basis_source': 'radial_basis', 'n_environment_basis_per_dim': 10, 'radial_knots': [0.0, 0.0, 0.30736593266463014, 0.6147318653292603, 0.9220977979938905,

,trial_index,trial_name,trial_base_seed,M,L,system_order,environment_variable,L_fit,L_full,case_name,...,E_res_relative,E_phi,E_phi_relative,E_env,E_env_relative,cond_A,rank,min_singular,max_singular,ridge_lambda
0,0,trial_000,909,1,51,2,velocity,51,201,trial_000_M_001_L_051,...,0.000006,0.000910,0.000952,0.026077,0.035143,inf,208,9.494975e-18,0.118687,1.000000e-08
1,0,trial_000,909,1,101,2,velocity,101,401,trial_000_M_001_L_101,...,0.000010,0.001627,0.001360,0.048567,0.070215,inf,208,7.893187e-18,0.098956,1.000000e-08
2,0,trial_000,909,1,201,2,velocity,201,801,trial_000_M_001_L_201,...,0.000009,0.001958,0.001588,0.053844,0.078088,inf,208,7.154559e-18,0.103255,1.000000e-08
3,0,trial_000,909,1,401,2,velocity,401,1601,trial_000_M_001_L_401,...,0.000007,0.009421,0.008367,0.069361,0.097192,inf,208,8.056635e-18,0.106488,1.000000e-08
4,0,trial_000,909,1,801,2,velocity,801,3201,trial_000_M_001_L_801,...,0.000005,0.001011,0.001144,0.047860,0.078162,inf,208,4.013600e-18,0.089569,1.000000e-08


,M,L,n_trials_requested,n_trials_recorded,n_successful_trials,n_failed_trials,trial_base_seed_count,trial_base_seed_mean,trial_base_seed_std,E_res_count,...,rank_std,min_singular_count,min_singular_mean,min_singular_std,max_singular_count,max_singular_mean,max_singular_std,ridge_lambda_count,ridge_lambda_mean,ridge_lambda_std
0,1,51,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,7.417021e-18,2.513351e-18,10,0.107701,0.014615,10,1.000000e-08,0.0
1,1,101,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,7.459545e-18,1.705395e-18,10,0.103612,0.013513,10,1.000000e-08,0.0
2,1,201,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,7.340856e-18,1.704451e-18,10,0.107496,0.010237,10,1.000000e-08,0.0
3,1,401,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,7.874781e-18,1.622673e-18,10,0.103406,0.013789,10,1.000000e-08,0.0
4,1,801,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,5.881493e-18,1.857811e-18,10,0.099035,0.008910,10,1.000000e-08,0.0


In [8]:
# --------------------------------------------------
# Load repeated-trial M/L sweep outputs
# --------------------------------------------------
# FIGURE_STAGE = "step_6C"

ml_eval_dir = (
    Path(config_6C.artifact_root)
    / config_6C.system_name
    / "evaluations"
    / "ML_sweep"
)

trial_metrics_path = (
    ml_eval_dir
    / "ML_sweep_trial_metrics.csv"
)

summary_metrics_path = (
    ml_eval_dir
    / "ML_sweep_metrics_mean_std.csv"
)

# One row per (trial, M, L).
df_ml_trials = pd.read_csv(trial_metrics_path)

# One row per (M, L), containing statistics across trials.
df_ml = pd.read_csv(summary_metrics_path)

print("Trial-level rows:", len(df_ml_trials))
print("Mean/std rows:", len(df_ml))

display(df_ml.head())

Trial-level rows: 250
Mean/std rows: 25


,M,L,n_trials_requested,n_trials_recorded,n_successful_trials,n_failed_trials,trial_base_seed_count,trial_base_seed_mean,trial_base_seed_std,E_res_count,...,rank_std,min_singular_count,min_singular_mean,min_singular_std,max_singular_count,max_singular_mean,max_singular_std,ridge_lambda_count,ridge_lambda_mean,ridge_lambda_std
0,1,51,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,7.417021e-18,2.513351e-18,10,0.107701,0.014615,10,1.000000e-08,0.0
1,1,101,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,7.459545e-18,1.705395e-18,10,0.103612,0.013513,10,1.000000e-08,0.0
2,1,201,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,7.340856e-18,1.704451e-18,10,0.107496,0.010237,10,1.000000e-08,0.0
3,1,401,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,7.874781e-18,1.622673e-18,10,0.103406,0.013789,10,1.000000e-08,0.0
4,1,801,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,5.881493e-18,1.857811e-18,10,0.099035,0.008910,10,1.000000e-08,0.0


In [9]:
required_summary_columns = [
    "M",
    "L",
    "n_trials_requested",
    "n_trials_recorded",
    "n_successful_trials",
    "n_failed_trials",
    "E_phi_relative_mean",
    "E_phi_relative_std",
    "E_env_relative_mean",
    "E_env_relative_std",
]

missing_columns = [
    column
    for column in required_summary_columns
    if column not in df_ml.columns
]

if missing_columns:
    raise ValueError(
        "Missing repeated-trial summary columns: "
        f"{missing_columns}"
    )

expected_summary_rows = (
    len(config_6C.M_values)
    * len(config_6C.L_values)
)

expected_trial_rows = (
    config_6C.n_trials
    * expected_summary_rows
)

print("Expected trial rows:", expected_trial_rows)
print("Observed trial rows:", len(df_ml_trials))

print("Expected summary rows:", expected_summary_rows)
print("Observed summary rows:", len(df_ml))

assert len(df_ml_trials) == expected_trial_rows
assert len(df_ml) == expected_summary_rows
assert not df_ml[["M", "L"]].duplicated().any()

Expected trial rows: 250
Observed trial rows: 250
Expected summary rows: 25
Observed summary rows: 25


# SPP

In [10]:
from ips_learning.analysis.ml_sweep_runner import MLSweepConfig, MLSweepRunner

In [11]:
SYSTEM_NAME = "spp"
PROFILE_NAME = "spp_velocity_grid_clumps"
INTERACTION_TYPE = "energy"
SYSTEM_ORDER = 2

FIT_END_TIME = 0.5
PREDICTION_END_TIME = 2.0
TRAIN_BEGIN_TIME = 0.0

In [12]:
M_VALUES = (1, 4, 16, 64, 256)
L_VALUES = (51, 101, 201, 401, 801)
# M_VALUES = (50,)
# L_VALUES = (76,)

In [13]:
config_6C = MLSweepConfig(
    system_name=SYSTEM_NAME,
    profile_name=PROFILE_NAME,
    interaction_type=INTERACTION_TYPE,
    system_order=SYSTEM_ORDER,
    artifact_root=ARTIFACTS,

    n_trials=N_TRIALS,
    base_seed=BASE_SEED,
    trial_seed_stride=TRIAL_SEED_STRIDE,

    N=10,
    D=2,
    t0=TRAIN_BEGIN_TIME,

    M_values=M_VALUES,
    L_values=L_VALUES,

    fit_horizon_end=FIT_END_TIME,
    prediction_horizon_end=PREDICTION_END_TIME,

    radial_degree=1,
    radial_n_basis=10,
    environment_degree=1,
    environment_n_basis_per_dim=10,

    solver_method="lstsq",
    ridge_lambda=1e-8,
    svd_tol=1e-12,

    kernel_grid_size=500,
    environment_grid_size=120,

    overwrite=OVERWRITE,
)

In [14]:
# --------------------------------------------------
# Run repeated-trial M/L sweep: SPP
# --------------------------------------------------
spp_runner = MLSweepRunner(config_6C)

# One row per (trial_index, M, L).
spp_trial_df = spp_runner.run()

spp_eval_dir = (
    Path(config_6C.artifact_root)
    / config_6C.system_name
    / "evaluations"
    / "ML_sweep"
)

spp_summary_path = (
    spp_eval_dir
    / "ML_sweep_metrics_mean_std.csv"
)

# One row per (M, L), containing mean/std across trials.
spp_summary_df = pd.read_csv(spp_summary_path)

print("Trial-level rows:", len(spp_trial_df))
print("Summary rows:", len(spp_summary_df))

display(spp_trial_df.head())
display(spp_summary_df.head())

Starting trial_000: 1/10
Trial base seed: 909
[1/250] trial_000, M=1, L=51
Generating/loading trial-specific dataset: trial_000, M=1, L=51
Simulating trajectory 1/1
Running trial-specific case: trial_000, M=1, L=51
fit_end_index:51
T_fit:   0.5
t:   (51,)
X:   (1, 51, 10, 2)
V: (1, 51, 10, 2)
LHS: (1, 51, 10, 2)
U0:  (1, 40)
interaction type: energy
condition number: 1.3105773215609354e+16
rank: 210
basis localization: {'system_name': 'spp', 'system_order': 2, 'interaction_type': 'energy', 'basis_stage': '2.2', 'environment_variable': 'velocity', 'r_min': 0.0, 'r_max': 8.136415209230925, 'env_min': [-2.8284271247461903, -2.7083792163618092], 'env_max': [2.578368272959014, 2.8284271247461903], 'radial_family': 'bspline', 'environment_family': 'bspline', 'radial_degree': 1, 'environment_degree': 1, 'n_radial_basis': 10, 'n_energy_basis': 10, 'n_alignment_basis': 10, 'energy_basis_source': 'radial_basis', 'alignment_basis_source': 'radial_basis', 'n_environment_basis_per_dim': 10, 'radial

,trial_index,trial_name,trial_base_seed,M,L,system_order,environment_variable,L_fit,L_full,case_name,...,E_res_relative,E_phi,E_phi_relative,E_env,E_env_relative,cond_A,rank,min_singular,max_singular,ridge_lambda
0,0,trial_000,909,1,51,2,velocity,51,201,trial_000_M_001_L_051,...,0.003070,2.014420,3.018033,33.345409,9.190843,inf,210,1.902910e-16,2.493911,1.000000e-08
1,0,trial_000,909,1,101,2,velocity,101,401,trial_000_M_001_L_101,...,0.005046,0.335764,0.213920,13.648231,3.399675,inf,210,1.879917e-17,0.224644,1.000000e-08
2,0,trial_000,909,1,201,2,velocity,201,801,trial_000_M_001_L_201,...,0.001884,0.250721,0.172653,3.669525,1.089543,inf,210,2.156808e-17,0.300005,1.000000e-08
3,0,trial_000,909,1,401,2,velocity,401,1601,trial_000_M_001_L_401,...,0.002048,0.954247,0.979000,1.789471,0.458621,inf,210,6.727733e-17,0.729223,1.000000e-08
4,0,trial_000,909,1,801,2,velocity,801,3201,trial_000_M_001_L_801,...,0.004103,1.243337,1.070644,6.191005,1.491321,inf,210,4.895214e-17,0.542308,1.000000e-08


,M,L,n_trials_requested,n_trials_recorded,n_successful_trials,n_failed_trials,trial_base_seed_count,trial_base_seed_mean,trial_base_seed_std,E_res_count,...,rank_std,min_singular_count,min_singular_mean,min_singular_std,max_singular_count,max_singular_mean,max_singular_std,ridge_lambda_count,ridge_lambda_mean,ridge_lambda_std
0,1,51,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,7.821193e-17,7.471852e-17,10,0.956560,0.931953,10,1.000000e-08,0.0
1,1,101,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,8.491829e-17,7.416358e-17,10,0.962134,0.818802,10,1.000000e-08,0.0
2,1,201,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,4.441120e-17,3.045769e-17,10,0.509393,0.322683,10,1.000000e-08,0.0
3,1,401,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,7.213429e-17,1.003622e-16,10,0.928317,1.131918,10,1.000000e-08,0.0
4,1,801,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,6.669841e-17,3.608843e-17,10,0.777921,0.374758,10,1.000000e-08,0.0


In [15]:
# --------------------------------------------------
# Load repeated-trial M/L sweep outputs
# --------------------------------------------------
# FIGURE_STAGE = "step_6C"

ml_eval_dir = (
    Path(config_6C.artifact_root)
    / config_6C.system_name
    / "evaluations"
    / "ML_sweep"
)

trial_metrics_path = (
    ml_eval_dir
    / "ML_sweep_trial_metrics.csv"
)

summary_metrics_path = (
    ml_eval_dir
    / "ML_sweep_metrics_mean_std.csv"
)

# One row per (trial, M, L).
df_ml_trials = pd.read_csv(trial_metrics_path)

# One row per (M, L), containing statistics across trials.
df_ml = pd.read_csv(summary_metrics_path)

print("Trial-level rows:", len(df_ml_trials))
print("Mean/std rows:", len(df_ml))

display(df_ml.head())

Trial-level rows: 250
Mean/std rows: 25


,M,L,n_trials_requested,n_trials_recorded,n_successful_trials,n_failed_trials,trial_base_seed_count,trial_base_seed_mean,trial_base_seed_std,E_res_count,...,rank_std,min_singular_count,min_singular_mean,min_singular_std,max_singular_count,max_singular_mean,max_singular_std,ridge_lambda_count,ridge_lambda_mean,ridge_lambda_std
0,1,51,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,7.821193e-17,7.471852e-17,10,0.956560,0.931953,10,1.000000e-08,0.0
1,1,101,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,8.491829e-17,7.416358e-17,10,0.962134,0.818802,10,1.000000e-08,0.0
2,1,201,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,4.441120e-17,3.045769e-17,10,0.509393,0.322683,10,1.000000e-08,0.0
3,1,401,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,7.213429e-17,1.003622e-16,10,0.928317,1.131918,10,1.000000e-08,0.0
4,1,801,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,6.669841e-17,3.608843e-17,10,0.777921,0.374758,10,1.000000e-08,0.0


In [16]:
required_summary_columns = [
    "M",
    "L",
    "n_trials_requested",
    "n_trials_recorded",
    "n_successful_trials",
    "n_failed_trials",
    "E_phi_relative_mean",
    "E_phi_relative_std",
    "E_env_relative_mean",
    "E_env_relative_std",
]

missing_columns = [
    column
    for column in required_summary_columns
    if column not in df_ml.columns
]

if missing_columns:
    raise ValueError(
        "Missing repeated-trial summary columns: "
        f"{missing_columns}"
    )

expected_summary_rows = (
    len(config_6C.M_values)
    * len(config_6C.L_values)
)

expected_trial_rows = (
    config_6C.n_trials
    * expected_summary_rows
)

print("Expected trial rows:", expected_trial_rows)
print("Observed trial rows:", len(df_ml_trials))

print("Expected summary rows:", expected_summary_rows)
print("Observed summary rows:", len(df_ml))

assert len(df_ml_trials) == expected_trial_rows
assert len(df_ml) == expected_summary_rows
assert not df_ml[["M", "L"]].duplicated().any()

Expected trial rows: 250
Observed trial rows: 250
Expected summary rows: 25
Observed summary rows: 25


# Kuramoto

In [17]:
from ips_learning.analysis.ml_sweep_runner import MLSweepConfig, MLSweepRunner

M_VALUES = (1, 4, 16, 64, 256)
L_VALUES = (51, 101, 201, 401, 801)
# M_VALUES = (20,)
# L_VALUES = (51,)

SYSTEM_NAME = "kuramoto"
PROFILE_NAME = "profile1"
INTERACTION_TYPE = "energy"
SYSTEM_ORDER = 1

FIT_END_TIME = 0.5
PREDICTION_END_TIME = 2.0
TRAIN_BEGIN_TIME = 0.0

In [18]:


config_6C = MLSweepConfig(
    system_name=SYSTEM_NAME,
    profile_name=PROFILE_NAME,
    interaction_type=INTERACTION_TYPE,
    system_order=SYSTEM_ORDER,
    artifact_root=ARTIFACTS,

    n_trials=N_TRIALS,
    base_seed=BASE_SEED,
    trial_seed_stride=TRIAL_SEED_STRIDE,

    N=10,
    D=1,
    t0=TRAIN_BEGIN_TIME,

    M_values=M_VALUES,
    L_values=L_VALUES,

    fit_horizon_end=FIT_END_TIME,
    prediction_horizon_end=PREDICTION_END_TIME,

    radial_degree=1,
    radial_n_basis=10,
    environment_degree=1,
    environment_n_basis_per_dim=10,

    solver_method="lstsq",
    ridge_lambda=1e-8,
    svd_tol=1e-12,

    kernel_grid_size=500,
    environment_grid_size=120,

    overwrite=OVERWRITE,
)



In [19]:
# --------------------------------------------------
# Run repeated-trial M/L sweep: Kuramoto
# --------------------------------------------------
kuramoto_runner = MLSweepRunner(config_6C)

# One row per (trial_index, M, L).
kuramoto_trial_df = kuramoto_runner.run()

kuramoto_eval_dir = (
    Path(config_6C.artifact_root)
    / config_6C.system_name
    / "evaluations"
    / "ML_sweep"
)

kuramoto_summary_path = (
    kuramoto_eval_dir
    / "ML_sweep_metrics_mean_std.csv"
)

# One row per (M, L), containing mean/std across trials.
kuramoto_summary_df = pd.read_csv(
    kuramoto_summary_path
)

print(
    "Trial-level rows:",
    len(kuramoto_trial_df),
)
print(
    "Summary rows:",
    len(kuramoto_summary_df),
)

display(kuramoto_trial_df.head())
display(kuramoto_summary_df.head())

Starting trial_000: 1/10
Trial base seed: 909
[1/250] trial_000, M=1, L=51
Generating/loading trial-specific dataset: trial_000, M=1, L=51
Simulating trajectory 1/1
Running trial-specific case: trial_000, M=1, L=51
fit_end_index:51
T_fit:   0.5
t:   (51,)
X:   (1, 51, 10, 1)
V: None
LHS: (1, 51, 10, 1)
U0:  (1, 10)
interaction type: energy
condition number: 61809867.47453985
rank: 20
basis localization: {'system_name': 'kuramoto', 'system_order': 1, 'interaction_type': 'energy', 'basis_stage': '2.2', 'environment_variable': 'position', 'r_min': 0.0, 'r_max': 5.043771614759337, 'env_min': [0.6992365383796636], 'env_max': [5.856319091215873], 'radial_family': 'bspline', 'environment_family': 'bspline', 'radial_degree': 1, 'environment_degree': 1, 'n_radial_basis': 10, 'n_energy_basis': 10, 'n_alignment_basis': 10, 'energy_basis_source': 'radial_basis', 'alignment_basis_source': 'radial_basis', 'n_environment_basis_per_dim': 10, 'radial_knots': [0.0, 0.0, 0.560419068306593, 1.120838136613

,trial_index,trial_name,trial_base_seed,M,L,system_order,environment_variable,L_fit,L_full,case_name,...,E_res_relative,E_phi,E_phi_relative,E_env,E_env_relative,cond_A,rank,min_singular,max_singular,ridge_lambda
0,0,trial_000,909,1,51,1,position,51,201,trial_000_M_001_L_051,...,0.003769,1.486680,1.094088,0.909803,28.073478,6.180987e+07,20,9.475157e-09,0.585658,1.000000e-08
1,0,trial_000,909,1,101,1,position,101,401,trial_000_M_001_L_101,...,0.004388,1.732202,1.221669,1.040582,31.695005,inf,20,3.656060e-21,0.706656,1.000000e-08
2,0,trial_000,909,1,201,1,position,201,801,trial_000_M_001_L_201,...,0.003057,1.248444,0.833394,0.778789,18.310798,5.772874e+18,20,8.237943e-20,0.475566,1.000000e-08
3,0,trial_000,909,1,401,1,position,401,1601,trial_000_M_001_L_401,...,0.001724,0.352667,0.248295,0.184304,5.866983,3.276179e+08,20,2.093187e-09,0.685765,1.000000e-08
4,0,trial_000,909,1,801,1,position,801,3201,trial_000_M_001_L_801,...,0.001384,0.246520,0.164921,0.144190,3.126872,2.986699e+19,20,1.449131e-20,0.432812,1.000000e-08


,M,L,n_trials_requested,n_trials_recorded,n_successful_trials,n_failed_trials,trial_base_seed_count,trial_base_seed_mean,trial_base_seed_std,E_res_count,...,rank_std,min_singular_count,min_singular_mean,min_singular_std,max_singular_count,max_singular_mean,max_singular_std,ridge_lambda_count,ridge_lambda_mean,ridge_lambda_std
0,1,51,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,2.023717e-08,2.929871e-08,10,0.739760,0.472673,10,1.000000e-08,0.0
1,1,101,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,7.367551e-08,1.748921e-07,10,0.855233,0.413712,10,1.000000e-08,0.0
2,1,201,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,3.492524e-09,6.873738e-09,10,0.626889,0.158475,10,1.000000e-08,0.0
3,1,401,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,2.683384e-08,5.188868e-08,10,0.753808,0.393735,10,1.000000e-08,0.0
4,1,801,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,1.143036e-08,2.468300e-08,10,0.873345,0.382719,10,1.000000e-08,0.0


In [20]:
# --------------------------------------------------
# Load repeated-trial M/L sweep outputs
# --------------------------------------------------
# FIGURE_STAGE = "step_6C"

ml_eval_dir = (
    Path(config_6C.artifact_root)
    / config_6C.system_name
    / "evaluations"
    / "ML_sweep"
)

trial_metrics_path = (
    ml_eval_dir
    / "ML_sweep_trial_metrics.csv"
)

summary_metrics_path = (
    ml_eval_dir
    / "ML_sweep_metrics_mean_std.csv"
)

# One row per (trial, M, L).
df_ml_trials = pd.read_csv(trial_metrics_path)

# One row per (M, L), containing statistics across trials.
df_ml = pd.read_csv(summary_metrics_path)

print("Trial-level rows:", len(df_ml_trials))
print("Mean/std rows:", len(df_ml))

display(df_ml.head())

Trial-level rows: 250
Mean/std rows: 25


,M,L,n_trials_requested,n_trials_recorded,n_successful_trials,n_failed_trials,trial_base_seed_count,trial_base_seed_mean,trial_base_seed_std,E_res_count,...,rank_std,min_singular_count,min_singular_mean,min_singular_std,max_singular_count,max_singular_mean,max_singular_std,ridge_lambda_count,ridge_lambda_mean,ridge_lambda_std
0,1,51,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,2.023717e-08,2.929871e-08,10,0.739760,0.472673,10,1.000000e-08,0.0
1,1,101,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,7.367551e-08,1.748921e-07,10,0.855233,0.413712,10,1.000000e-08,0.0
2,1,201,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,3.492524e-09,6.873738e-09,10,0.626889,0.158475,10,1.000000e-08,0.0
3,1,401,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,2.683384e-08,5.188868e-08,10,0.753808,0.393735,10,1.000000e-08,0.0
4,1,801,10,10,10,0,10,450000909.0,3.027650e+08,10,...,0.0,10,1.143036e-08,2.468300e-08,10,0.873345,0.382719,10,1.000000e-08,0.0


In [21]:
required_summary_columns = [
    "M",
    "L",
    "n_trials_requested",
    "n_trials_recorded",
    "n_successful_trials",
    "n_failed_trials",
    "E_phi_relative_mean",
    "E_phi_relative_std",
    "E_env_relative_mean",
    "E_env_relative_std",
]

missing_columns = [
    column
    for column in required_summary_columns
    if column not in df_ml.columns
]

if missing_columns:
    raise ValueError(
        "Missing repeated-trial summary columns: "
        f"{missing_columns}"
    )

expected_summary_rows = (
    len(config_6C.M_values)
    * len(config_6C.L_values)
)

expected_trial_rows = (
    config_6C.n_trials
    * expected_summary_rows
)

print("Expected trial rows:", expected_trial_rows)
print("Observed trial rows:", len(df_ml_trials))

print("Expected summary rows:", expected_summary_rows)
print("Observed summary rows:", len(df_ml))

assert len(df_ml_trials) == expected_trial_rows
assert len(df_ml) == expected_summary_rows
assert not df_ml[["M", "L"]].duplicated().any()

Expected trial rows: 250
Observed trial rows: 250
Expected summary rows: 25
Observed summary rows: 25
